# Device 1 - Capturing images
1. Controller with camera: https://amzn.to/3w7MiV1
2. Micro USB - USB cable: https://amzn.to/4aSkZNp
3. Power bank: https://amzn.to/3UhFdcn
4. Micro SD card: https://www.amazon.com/SanDisk-microSDHC-Memory-SDSDQ-004G-Packaging/dp/B00FM1JGHK

# Device 2 - Inferencing device
1. Raspberry pi kit
2. Touch screen display: https://www.amazon.com/Touchscreen-Raspberry-Compatible-Raspbian-RetroPie/dp/B091FYFNV8
3. Power bank

# Device 1: Implementation

##Capturing images and saving them to SD card

### Capture pictures on reboot using reset button

In [ ]:
#include "esp_camera.h"
#include "Arduino.h"
#include "FS.h"                // SD Card ESP32
#include "SD_MMC.h"            // SD Card ESP32
#include "soc/soc.h"           // Disable brownour problems
#include "soc/rtc_cntl_reg.h"  // Disable brownour problems
#include "driver/rtc_io.h"
#include <EEPROM.h>            // read and write from flash memory

// define the number of bytes you want to access
#define EEPROM_SIZE 1

// Pin definition for CAMERA_MODEL_AI_THINKER
#define PWDN_GPIO_NUM     32
#define RESET_GPIO_NUM    -1
#define XCLK_GPIO_NUM      0
#define SIOD_GPIO_NUM     26
#define SIOC_GPIO_NUM     27

#define Y9_GPIO_NUM       35
#define Y8_GPIO_NUM       34
#define Y7_GPIO_NUM       39
#define Y6_GPIO_NUM       36
#define Y5_GPIO_NUM       21
#define Y4_GPIO_NUM       19
#define Y3_GPIO_NUM       18
#define Y2_GPIO_NUM        5
#define VSYNC_GPIO_NUM    25
#define HREF_GPIO_NUM     23
#define PCLK_GPIO_NUM     22

int pictureNumber = 0;

void setup() {
  WRITE_PERI_REG(RTC_CNTL_BROWN_OUT_REG, 0); //disable brownout detector

  Serial.begin(115200);
  //Serial.setDebugOutput(true);
  //Serial.println();

  camera_config_t config;
  config.ledc_channel = LEDC_CHANNEL_0;
  config.ledc_timer = LEDC_TIMER_0;
  config.pin_d0 = Y2_GPIO_NUM;
  config.pin_d1 = Y3_GPIO_NUM;
  config.pin_d2 = Y4_GPIO_NUM;
  config.pin_d3 = Y5_GPIO_NUM;
  config.pin_d4 = Y6_GPIO_NUM;
  config.pin_d5 = Y7_GPIO_NUM;
  config.pin_d6 = Y8_GPIO_NUM;
  config.pin_d7 = Y9_GPIO_NUM;
  config.pin_xclk = XCLK_GPIO_NUM;
  config.pin_pclk = PCLK_GPIO_NUM;
  config.pin_vsync = VSYNC_GPIO_NUM;
  config.pin_href = HREF_GPIO_NUM;
  config.pin_sscb_sda = SIOD_GPIO_NUM;
  config.pin_sscb_scl = SIOC_GPIO_NUM;
  config.pin_pwdn = PWDN_GPIO_NUM;
  config.pin_reset = RESET_GPIO_NUM;
  config.xclk_freq_hz = 20000000;
  config.pixel_format = PIXFORMAT_JPEG;

  if(psramFound()){
    config.frame_size = FRAMESIZE_UXGA; // FRAMESIZE_ + QVGA|CIF|VGA|SVGA|XGA|SXGA|UXGA
    config.jpeg_quality = 10;
    config.fb_count = 2;
  } else {
    config.frame_size = FRAMESIZE_SVGA;
    config.jpeg_quality = 12;
    config.fb_count = 1;
  }

  // Init Camera
  esp_err_t err = esp_camera_init(&config);
  if (err != ESP_OK) {
    Serial.printf("Camera init failed with error 0x%x", err);
    return;
  }

  //Serial.println("Starting SD Card");
  if(!SD_MMC.begin()){
    Serial.println("SD Card Mount Failed");
    return;
  }

  uint8_t cardType = SD_MMC.cardType();
  if(cardType == CARD_NONE){
    Serial.println("No SD Card attached");
    return;
  }

  camera_fb_t * fb = NULL;

  // Take Picture with Camera
  fb = esp_camera_fb_get(); //frame buffer
  if(!fb) {
    Serial.println("Camera capture failed");
    return;
  }
  // initialize EEPROM with predefined size
  EEPROM.begin(EEPROM_SIZE);
  pictureNumber = EEPROM.read(0) + 1;

  // Path where new picture will be saved in SD Card
  String path = "/picture" + String(pictureNumber) +".jpg";

  fs::FS &fs = SD_MMC;
  Serial.printf("Picture file name: %s\n", path.c_str());

  File file = fs.open(path.c_str(), FILE_WRITE);
  if(!file){
    Serial.println("Failed to open file in writing mode");
  }
  else {
    file.write(fb->buf, fb->len); // payload (image), payload length
    Serial.printf("Saved file to path: %s\n", path.c_str());
    EEPROM.write(0, pictureNumber);
    EEPROM.commit();
  }
  file.close();
  esp_camera_fb_return(fb);

  // Turns off the ESP32-CAM white on-board LED (flash) connected to GPIO 4
  pinMode(4, OUTPUT);
  digitalWrite(4, LOW);
  rtc_gpio_hold_en(GPIO_NUM_4);

  delay(2000);
  Serial.println("Going to sleep now");
  delay(2000);
  esp_deep_sleep_start();
  Serial.println("This will never be printed");
}

void loop() {

}

### Capture in a loop with file name as timestamp - Using WiFi and NTP

In [ ]:
#include "esp_camera.h"
#include "FS.h"                // SD Card ESP32
#include "SD_MMC.h"            // SD Card ESP32
#include "soc/soc.h"           // Disable brownout problems
#include "soc/rtc_cntl_reg.h"  // Disable brownout problems
#include "driver/rtc_io.h"
#include <WiFi.h>
#include "time.h"


// REPLACE WITH YOUR NETWORK CREDENTIALS
const char* ssid = "Extensible24";
const char* password = "22011995";


// REPLACE WITH YOUR TIMEZONE STRING: https://github.com/nayarsystems/posix_tz_db/blob/master/zones.csv
String myTimezone ="PST-8";


// Pin definition for CAMERA_MODEL_AI_THINKER
// Change pin definition if you're using another ESP32 camera module
#define PWDN_GPIO_NUM     32
#define RESET_GPIO_NUM    -1
#define XCLK_GPIO_NUM      0
#define SIOD_GPIO_NUM     26
#define SIOC_GPIO_NUM     27
#define Y9_GPIO_NUM       35
#define Y8_GPIO_NUM       34
#define Y7_GPIO_NUM       39
#define Y6_GPIO_NUM       36
#define Y5_GPIO_NUM       21
#define Y4_GPIO_NUM       19
#define Y3_GPIO_NUM       18
#define Y2_GPIO_NUM        5
#define VSYNC_GPIO_NUM    25
#define HREF_GPIO_NUM     23
#define PCLK_GPIO_NUM     22


// Stores the camera configuration parameters
camera_config_t config;


// Initializes the camera
void configInitCamera(){
  config.ledc_channel = LEDC_CHANNEL_0;
  config.ledc_timer = LEDC_TIMER_0;
  config.pin_d0 = Y2_GPIO_NUM;
  config.pin_d1 = Y3_GPIO_NUM;
  config.pin_d2 = Y4_GPIO_NUM;
  config.pin_d3 = Y5_GPIO_NUM;
  config.pin_d4 = Y6_GPIO_NUM;
  config.pin_d5 = Y7_GPIO_NUM;
  config.pin_d6 = Y8_GPIO_NUM;
  config.pin_d7 = Y9_GPIO_NUM;
  config.pin_xclk = XCLK_GPIO_NUM;
  config.pin_pclk = PCLK_GPIO_NUM;
  config.pin_vsync = VSYNC_GPIO_NUM;
  config.pin_href = HREF_GPIO_NUM;
  config.pin_sscb_sda = SIOD_GPIO_NUM;
  config.pin_sscb_scl = SIOC_GPIO_NUM;
  config.pin_pwdn = PWDN_GPIO_NUM;
  config.pin_reset = RESET_GPIO_NUM;
  config.xclk_freq_hz = 20000000;
  config.pixel_format = PIXFORMAT_JPEG; //YUV422,GRAYSCALE,RGB565,JPEG
  config.grab_mode = CAMERA_GRAB_LATEST;


  // Select lower framesize if the camera doesn't support PSRAM
  if(psramFound()){
    config.frame_size = FRAMESIZE_UXGA; // FRAMESIZE_ + QVGA|CIF|VGA|SVGA|XGA|SXGA|UXGA
    config.jpeg_quality = 10; //0-63 lower number means higher quality
    config.fb_count = 1;
  } else {
    config.frame_size = FRAMESIZE_SVGA;
    config.jpeg_quality = 12;
    config.fb_count = 1;
  }

  // Initialize the Camera
  esp_err_t err = esp_camera_init(&config);
  if (err != ESP_OK) {
    Serial.printf("Camera init failed with error 0x%x", err);
    return;
  }
}


// Connect to wifi
void  initWiFi(){
  WiFi.begin(ssid, password);
  Serial.println("Connecting Wifi");
  while (WiFi.status() != WL_CONNECTED) {
    Serial.print(".");
    delay(500);
  }
}


// Function to set timezone
void setTimezone(String timezone){
  Serial.printf("  Setting Timezone to %s\n",timezone.c_str());
  setenv("TZ",timezone.c_str(),1);  //  Now adjust the TZ.  Clock settings are adjusted to show the new local time
  tzset();
}


// Connect to NTP server and adjust timezone
void initTime(String timezone){
  struct tm timeinfo;
  Serial.println("Setting up time");
  configTime(0, 0, "pool.ntp.org");    // First connect to NTP server, with 0 TZ offset
  if(!getLocalTime(&timeinfo)){
    Serial.println(" Failed to obtain time");
    return;
  }
  Serial.println("Got the time from NTP");
  // Now we can set the real timezone
  setTimezone(timezone);
}


// Get the picture filename based on the current ime
String getPictureFilename(){
  struct tm timeinfo;
  if(!getLocalTime(&timeinfo)){
    Serial.println("Failed to obtain time");
    return "";
  }
  char timeString[20];
  strftime(timeString, sizeof(timeString), "%Y-%m-%d_%H-%M-%S", &timeinfo);
  Serial.println(timeString);
  String filename = "/picture_" + String(timeString) +".jpg";
  return filename;
}


// Initialize the micro SD card
void initMicroSDCard(){
  // Start Micro SD card
  Serial.println("Starting SD Card");
  if(!SD_MMC.begin()){
    Serial.println("SD Card Mount Failed");
    return;
  }
  uint8_t cardType = SD_MMC.cardType();
  if(cardType == CARD_NONE){
    Serial.println("No SD Card attached");
    return;
  }
}


// Take photo and save to microSD card
void takeSavePhoto(){
  // Take Picture with Camera
  camera_fb_t * fb = esp_camera_fb_get();

  //Uncomment the following lines if you're getting old pictures
  //esp_camera_fb_return(fb); // dispose the buffered image
  //fb = NULL; // reset to capture errors
  //fb = esp_camera_fb_get();

  if(!fb) {
    Serial.println("Camera capture failed");
    delay(1000);
    ESP.restart();
  }


  // Path where new picture will be saved in SD Card
  String path = getPictureFilename();
  Serial.printf("Picture file name: %s\n", path.c_str());

  // Save picture to microSD card
  fs::FS &fs = SD_MMC;
  File file = fs.open(path.c_str(),FILE_WRITE);
  if(!file){
    Serial.printf("Failed to open file in writing mode");
  }
  else {
    file.write(fb->buf, fb->len); // payload (image), payload length
    Serial.printf("Saved: %s\n", path.c_str());
  }
  file.close();
  esp_camera_fb_return(fb);
}


void setup() {
  WRITE_PERI_REG(RTC_CNTL_BROWN_OUT_REG, 0); // disable brownout detector


  Serial.begin(115200);
  delay(2000);


  // Initialize Wi-Fi
  initWiFi();
  // Initialize time with timezone
  initTime(myTimezone);
  // Initialize the camera
  Serial.print("Initializing the camera module...");
  configInitCamera();
  Serial.println("Ok!");
  // Initialize MicroSD
  Serial.print("Initializing the MicroSD card module... ");
  initMicroSDCard();
}


void loop() {
  // Take and Save Photo
  takeSavePhoto();
  delay(10000);
}

## Updated code - Uses manual time setting

In [ ]:
#include "esp_camera.h"
#include "FS.h"                // SD Card ESP32
#include "SD_MMC.h"            // SD Card ESP32
#include "soc/soc.h"           // Disable brownout problems
#include "soc/rtc_cntl_reg.h"  // Disable brownout problems
#include "driver/rtc_io.h"
#include "time.h"

// Pin definition for CAMERA_MODEL_AI_THINKER
// Change pin definition if you're using another ESP32 camera module
#define PWDN_GPIO_NUM     32
#define RESET_GPIO_NUM    -1
#define XCLK_GPIO_NUM      0
#define SIOD_GPIO_NUM     26
#define SIOC_GPIO_NUM     27
#define Y9_GPIO_NUM       35
#define Y8_GPIO_NUM       34
#define Y7_GPIO_NUM       39
#define Y6_GPIO_NUM       36
#define Y5_GPIO_NUM       21
#define Y4_GPIO_NUM       19
#define Y3_GPIO_NUM       18
#define Y2_GPIO_NUM        5
#define VSYNC_GPIO_NUM    25
#define HREF_GPIO_NUM     23
#define PCLK_GPIO_NUM     22

//# define BUTTON_PIN        0   // Pin connected to the button
#define FLASH_LED_PIN      4   // Flash LED Pin (usually GPIO 4)

// Camera configuration
camera_config_t config;

// Function to initialize the camera
void configInitCamera() {
  config.ledc_channel = LEDC_CHANNEL_0;
  config.ledc_timer = LEDC_TIMER_0;
  config.pin_d0 = Y2_GPIO_NUM;
  config.pin_d1 = Y3_GPIO_NUM;
  config.pin_d2 = Y4_GPIO_NUM;
  config.pin_d3 = Y5_GPIO_NUM;
  config.pin_d4 = Y6_GPIO_NUM;
  config.pin_d5 = Y7_GPIO_NUM;
  config.pin_d6 = Y8_GPIO_NUM;
  config.pin_d7 = Y9_GPIO_NUM;
  config.pin_xclk = XCLK_GPIO_NUM;
  config.pin_pclk = PCLK_GPIO_NUM;
  config.pin_vsync = VSYNC_GPIO_NUM;
  config.pin_href = HREF_GPIO_NUM;
  config.pin_sscb_sda = SIOD_GPIO_NUM;
  config.pin_sscb_scl = SIOC_GPIO_NUM;
  config.pin_pwdn = PWDN_GPIO_NUM;
  config.pin_reset = RESET_GPIO_NUM;
  config.xclk_freq_hz = 20000000;
  config.pixel_format = PIXFORMAT_JPEG;

  if (psramFound()) {
    config.frame_size = FRAMESIZE_UXGA;
    config.jpeg_quality = 10;
    config.fb_count = 1;
  } else {
    config.frame_size = FRAMESIZE_SVGA;
    config.jpeg_quality = 12;
    config.fb_count = 1;
  }

  // Initialize the camera
  esp_err_t err = esp_camera_init(&config);
  if (err != ESP_OK) {
    Serial.printf("Camera init failed with error 0x%x", err);
    return;
  }
}

// Manually set the time
void setupManualTime() {
  struct tm timeinfo;
  timeinfo.tm_year = 2024 - 1900; // Year since 1900
  timeinfo.tm_mon = 9;             // Month (0-11, where 0 = January)
  timeinfo.tm_mday = 19;           // Day of the month
  timeinfo.tm_hour = 15;           // Hour (0-23)
  timeinfo.tm_min = 30;            // Minute (0-59)
  timeinfo.tm_sec = 0;             // Second (0-59)

  time_t t = mktime(&timeinfo);
  if (t == -1) {
    Serial.println("Failed to set time using mktime");
    return;
  }

  struct timeval now = { .tv_sec = t };
  settimeofday(&now, NULL);
}

// Get the picture filename based on the current time
String getPictureFilename() {
  struct tm timeinfo;
  if (!getLocalTime(&timeinfo)) {
    Serial.println("Failed to obtain time");
    return "";
  }
  char timeString[20];
  strftime(timeString, sizeof(timeString), "%Y-%m-%d_%H-%M-%S", &timeinfo);
  Serial.println(timeString);
  String filename = "/picture_" + String(timeString) + ".jpg";
  return filename;
}

// Initialize the micro SD card
void initMicroSDCard() {
  // Start Micro SD card
  Serial.println("Starting SD Card");
  if (!SD_MMC.begin()) {
    Serial.println("SD Card Mount Failed");
    return;
  }
  uint8_t cardType = SD_MMC.cardType();
  if (cardType == CARD_NONE) {
    Serial.println("No SD Card attached");
    return;
  }
}

// Capture image and save to SD card
void captureAndSaveImage() {
  // Turn on the flash
  digitalWrite(FLASH_LED_PIN, HIGH);
  delay(100); // Give a little time for illumination

  // Take Picture with Camera
  camera_fb_t *fb = esp_camera_fb_get();
  if (!fb) {
    Serial.println("Camera capture failed");
    digitalWrite(FLASH_LED_PIN, LOW); // Turn off the flash
    delay(1000);
    return;
  }

  // Path where new picture will be saved in SD Card
  String path = getPictureFilename();
  Serial.printf("Picture file name: %s\n", path.c_str());

  // Save picture to microSD card
  fs::FS &fs = SD_MMC;
  File file = fs.open(path.c_str(), FILE_WRITE);
  if (!file) {
    Serial.printf("Failed to open file in writing mode");
  } else {
    file.write(fb->buf, fb->len); // payload (image), payload length
    Serial.printf("Saved: %s\n", path.c_str());
  }
  file.close();
  esp_camera_fb_return(fb);

  // Turn off the flash
  digitalWrite(FLASH_LED_PIN, LOW);
}

void setup() {
  WRITE_PERI_REG(RTC_CNTL_BROWN_OUT_REG, 0); // disable brownout detector

  Serial.begin(115200);
  delay(2000);

  pinMode(BUTTON_PIN, INPUT_PULLUP);
  pinMode(FLASH_LED_PIN, OUTPUT); // Set the flash LED pin as an output
  digitalWrite(FLASH_LED_PIN, LOW); // Turn off the flash initially

  // Manually set the initial time
  setupManualTime();

  // Initialize the camera
  Serial.print("Initializing the camera module...");
  configInitCamera();
  Serial.println("Ok!");

  // Initialize MicroSD
  Serial.print("Initializing the MicroSD card module... ");
  initMicroSDCard();
}

void loop() {
  // Check if button is pressed
  if (digitalRead(BUTTON_PIN) == LOW) {
    delay(200); // Debounce delay
    if (digitalRead(BUTTON_PIN) == LOW) {
      captureAndSaveImage();
    }
  }
}

## Change Camera settings

In [ ]:
sensor_t * s = esp_camera_sensor_get();
s->set_brightness(s, 0);     // -2 to 2
s->set_contrast(s, 0);       // -2 to 2
s->set_gainceiling(s, (gainceiling_t)0);  // 0 to 6

# Device 2: Implementation

## Steps required
1. Setup Raspberry pi
  - Flashing OS
  - SSH and VNC setup
  - Installing required packages: Opencv, requests, numpy, pandas

2. Write a program to load an image, send to the model, and get response
3. Save the responses in a csv file
4. Detect the USB storage device event
  - copy all the files in a folder
  - Pass images in the storage device through the model and get responses
  - Save the results in a csv file
    - Extract the time stamp from the image file_name
    - Format timestamp as actual time, and save to timestamp column
    - Save the file name and responses in their respective column

In [ ]:
sudo apt-get install libssl-dev

In [ ]:
sudo apt-get update
sudo apt-get upgrade
sudo apt-get install python-opencv

## Get predictions and save to a csv file

In [ ]:
import requests
import base64
import os
import time
import pandas as pd

filepath = 'responses.csv'

# Check if the file exists
if os.path.exists(filepath):
   # Load the CSV file into a DataFrame
   df = pd.read_csv(filepath)
   print("File loaded successfully.")
else:
  # Create an empty DataFrame with specified columns
  df = pd.DataFrame(columns = ['timestamp', 'filename', 'response'])
  # Save the empty DataFrame as a CSV file
  df.to_csv(filepath, index=False)

categories = ["Bleached Corals", "Healthy Corals"]
def get_prediction(image_data):
  url = 'https://askai.aiclub.world/cecc4b02-43c5-4bc2-8906-2e8c7120d43e'
  r = requests.post(url, data=image_data)
  response = r.json()['predicted_label']
  print(r.json())
  print(categories[response])
  return categories[response]
image_path = "/home/robynbernabe/HealthyandBleachedCorals/Test/healthy_corals/"
file_names = []
for root, dirs, files in os.walk(image_path):
  for file_name in files:
    file_names.append(file_name)
    with open(image_path + file_name, "rb") as image:
      payload = base64.b64encode(image.read())
    response = get_prediction(payload)
    timestamp = int(time.time())
    df.loc[len(df)+1] = [timestamp, image_path + file_name, response]
    df.to_csv(filepath, index = False)

## Create a dashboard to visulaize the table

In [ ]:
import dash
from dash import dcc, html
import dash_table
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('data.csv')

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout of the app
app.layout = html.Div([
    html.H1("CSV Data Table"),
    dash_table.DataTable(
        id='table',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'},
    )
])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)

In [ ]:
import dash
import dash_table
import pandas as pd
import dash_bootstrap_components as dbc
from dash import html

# Load the CSV file into a DataFrame
df = pd.read_csv('data.csv')

# Initialize the Dash app with a Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define the layout of the app
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("CSV Data Table", className="text-center"), className="mb-4 mt-4")
    ]),
    dbc.Row([
        dbc.Col(
            dash_table.DataTable(
                id='table',
                columns=[{"name": i, "id": i} for i in df.columns],
                data=df.to_dict('records'),
                style_table={'overflowX': 'auto'},
                style_cell={'textAlign': 'left'},
                style_header={
                    'backgroundColor': 'rgb(30, 30, 30)',
                    'color': 'white'
                },
                style_data={
                    'backgroundColor': 'rgb(50, 50, 50)',
                    'color': 'white'
                },
                filter_action='native',
                sort_action='native',
                page_action='native',
                page_size=10
            )
        )
    ])
])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)

### Dashboard with bar chart

In [ ]:
pip install plotly

In [ ]:
import dash
import dash_table
import pandas as pd
import dash_bootstrap_components as dbc
from dash import html, dcc
import plotly.express as px

# Load the CSV file into a DataFrame
df = pd.read_csv('responses.csv')

# Calculate the proportions of each type
proportion_df = df['response'].value_counts(normalize=True).reset_index()
proportion_df.columns = ['response', 'proportion']

# Create a bar chart using Plotly
fig = px.bar(proportion_df, x='response', y='proportion', color='response',
             title='Proportion of Healthy and Bleached Corals',
             labels={'response': 'Coral Type', 'proportion': 'Proportion'},
             color_discrete_map={'Healthy Corals': 'green', 'Bleached Corals': 'red'})

# Initialize the Dash app with a Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Define the layout of the app
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("CSV Data Table and Coral Proportions", className="text-center"), className="mb-4 mt-4")
    ]),
    dbc.Row([
        dbc.Col(
            dash_table.DataTable(
                id='table',
                columns=[{"name": i, "id": i} for i in df.columns],
                data=df.to_dict('records'),
                style_table={'overflowX': 'auto'},
                style_cell={'textAlign': 'left'},
                style_header={
                    'backgroundColor': 'rgb(30, 30, 30)',
                    'color': 'white'
                },
                style_data={
                    'backgroundColor': 'rgb(50, 50, 50)',
                    'color': 'white'
                },
                filter_action='native',
                sort_action='native',
                page_action='native',
                page_size=10
            )
        )
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig), className="mt-4")
    ])
])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)

## Steps to detect USB event and list all files
Mount point = /media/robynbernabe/

In [ ]:
pip install watchdog

In [ ]:
import time
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import os

class USBHandler(FileSystemEventHandler):
    def on_created(self, event):
        if event.is_directory:
            # Wait to ensure the directory is fully mounted and available
            time.sleep(5)
            print(f"Directory created: {event.src_path}")
            list_files_on_usb(event.src_path)

def list_files_on_usb(mount_point):
    files_list = []
    for root, dirs, files in os.walk(mount_point):
        for file in files:
            if file.lower().endswith('.jpg'):  # Only add files with .jpg extension
                file_path = os.path.join(root, file)
                files_list.append(file_path)
    print("Files detected in the USB drive:")
    for file in files_list:
        print(file)

def monitor_mount_point(mount_point='/media/robynbernabe'):
    event_handler = USBHandler()
    observer = Observer()
    observer.schedule(event_handler, path=mount_point, recursive=True)
    observer.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        observer.stop()
    observer.join()

if __name__ == "__main__":
    monitor_mount_point()

## Sending files to the model


In [ ]:
import requests
import base64
import os
import time
import pandas as pd

filepath = 'responses.csv'
# Check if the file exists
if os.path.exists(filepath):
   # Load the CSV file into a DataFrame
   df = pd.read_csv(filepath)
   print("File loaded successfully.")
else:
  # Create an empty DataFrame with specified columns
  df = pd.DataFrame(columns = ['timestamp', 'filename', 'response'])
  # Save the empty DataFrame as a CSV file
  df.to_csv(filepath, index=False)

categories = ["Bleached Corals", "Healthy Corals"]
def get_prediction(image_data):
  url = 'https://askai.aiclub.world/cecc4b02-43c5-4bc2-8906-2e8c7120d43e'
  r = requests.post(url, data=image_data)
  response = r.json()['predicted_label']
  print(r.json())
  print(categories[response])
  return categories[response]
image_path = "/home/robynbernabe/HealthyandBleachedCorals/Test/healthy_corals/"
file_names = []
for root, dirs, files in os.walk(image_path):
  for file_name in files:
    file_names.append(file_name)
    with open(image_path + file_name, "rb") as image:
      payload = base64.b64encode(image.read())
    response = get_prediction(payload)
    timestamp = int(time.time())
    df.loc[len(df)+1] = [timestamp, file_name, response]
    df.to_csv(filepath, index = False)


## Combining detection of flash drive and sending files for prediction

In [ ]:
import time
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import os
import pandas as pd

filepath = 'responses.csv'
# Check if the file exists
if os.path.exists(filepath):
   # Load the CSV file into a DataFrame
   df = pd.read_csv(filepath)
   print("File loaded successfully.")
else:
  # Create an empty DataFrame with specified columns
  df = pd.DataFrame(columns = ['timestamp', 'filename', 'response'])
  # Save the empty DataFrame as a CSV file
  df.to_csv(filepath, index=False)

categories = ["Bleached Corals", "Healthy Corals"]
def get_prediction(image_data):
  url = 'https://askai.aiclub.world/cecc4b02-43c5-4bc2-8906-2e8c7120d43e'
  r = requests.post(url, data=image_data)
  response = r.json()['predicted_label']
  print(r.json())
  print(categories[response])
  return categories[response]

image_path = "../../media/robynbernabe/"
destination = "/home/robynbernabe/CoralHealthProject/captureimages/"

class USBHandler(FileSystemEventHandler):
    def on_created(self, event):
        if event.is_directory:
            # Wait to ensure the directory is fully mounted and available
            time.sleep(1)
            print(f"Directory created: {event.src_path}")
            list_files_on_usb(event.src_path)

def list_files_on_usb(mount_point):
    files_list = []
    for root, dirs, files in os.walk(mount_point):
        for file in files:
            file_path = os.path.join(root, file)
            files_list.append(file_path)
            with open(image_path + file, "rb") as image:
              payload = base64.b64encode(image.read())
            response = get_prediction(payload)
            timestamp = int(time.time())
            df.loc[len(df)+1] = [timestamp, destination+file, response]
            df.to_csv(filepath, index = False)

            # Move the file to a folder in Raspberry pi
            os.rename(image_path + file, destination)

    print("Files detected in the USB drive:")
    for file in files_list:
        print(file)

def monitor_mount_point(mount_point='/media/robynbernabe/'):
    event_handler = USBHandler()
    observer = Observer()
    observer.schedule(event_handler, path=mount_point, recursive=True)
    observer.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        observer.stop()
    observer.join()

if __name__ == "__main__":
    monitor_mount_point()



## Final dashboard to view results and preview the images
1. Show the results in a table.
2. Show the number of files in each category
3. User should be able to preview the images.
4. Run the dashboard as soon as raspberry pi starts.

In [ ]:
!pip install streamlit pandas

In [ ]:
import streamlit as st
import pandas as pd
from PIL import Image
import os

# Load the CSV file
csv_file = 'responses.csv'  # Replace with the path to your CSV file
df = pd.read_csv(csv_file)

st.title("Coral Image Predictions Dashboard")

# Count occurrences of each coral type
good_coral_count = df[df['response'] == 'Healthy Corals'].shape[0]
bleached_coral_count = df[df['response'] == 'Bleached Corals'].shape[0]

# Display coral counts at the top
st.write(f"### Coral Count Summary:")
st.write(f"**Good Coral**: {good_coral_count}")
st.write(f"**Bleached Coral**: {bleached_coral_count}")

# Show the dataframe as a table
st.write("### Predictions Table:")
st.dataframe(df)

# Add a dropdown to select file paths from the CSV
selected_image_path = st.selectbox("Select an image to view:", df['filename'])

# Display the selected image
if os.path.exists(selected_image_path):
    st.image(Image.open(selected_image_path), caption=selected_image_path)
else:
    st.write("Image not found!")

In [ ]:
streamlit run dashboard.py

## Run the dashboard on startup

In [ ]:
sudo apt-get install xautomation

In [ ]:
nano ~/start_streamlit.sh

In [ ]:
#!/bin/bash

# Step 1: Start the Streamlit app in the background
streamlit run /home/robynbernabe/CoralHealthProject/dashboard.py &

# Step 2: Wait for Streamlit to start
sleep 5  # Adjust this if needed to give Streamlit enough time to start

# Step 3: Open Chromium in full-screen mode, pointing to the local Streamlit server
chromium-browser --start-fullscreen --app=http://localhost:8501 &

# Step 4: Send an F11 keypress to ensure Chromium is in full-screen mode
sleep 2
xte "key F11"

In [ ]:
#!/bin/bash

# Log file
LOGFILE=~/script_log.txt
echo "Starting script at $(date)" >> $LOGFILE

# Step 1: Start the Streamlit app in the background
echo "Starting Streamlit..." >> $LOGFILE
streamlit run /home/robynbernabe/CoralHealthProject/dashboard.py &
STREAMLIT_PID=$!

# Step 2: Wait for Streamlit to start
sleep 5
echo "Waiting for Streamlit to start..." >> $LOGFILE

# Check if Streamlit is running
if ps -p $STREAMLIT_PID > /dev/null
then
    echo "Streamlit is running..." >> $LOGFILE
else
    echo "Streamlit failed to start." >> $LOGFILE
    exit 1
fi

# Step 3: Open Chromium in full-screen mode
echo "Launching Chromium..." >> $LOGFILE
chromium-browser --start-fullscreen --app=http://localhost:8501 &
sleep 2

# Step 4: Send an F11 keypress
echo "Sending F11 keypress..." >> $LOGFILE
xte "key F11"

echo "Script completed at $(date)" >> $LOGFILE

In [ ]:
chmod +x ~/start_streamlit.sh

In [ ]:
sudo crontab -e

In [ ]:
@reboot sh /home/robynbernabe/start_streamlit.sh >/home/robynbernabe/logs/cronlog 2>&1

# Mechanical casing for device 1

1. Dimensions of power bank: l x b x h
3. Dimension of controller: l x b x h
